# 🧠 3. Generación de Embeddings con ProtFlash

Este notebook corresponde a la **cuarta fase del pipeline de datos de ProtFlash**. Su objetivo es transformar las secuencias consolidadas en representaciones vectoriales densas (embeddings) utilizando el modelo preentrenado ProtFlash, listas para consumo en la etapa de entrenamiento supervisado.

## 📦 Instalación de dependencias

In [1]:
!pip install --upgrade ProtFlash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 106.1 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 req

## 🔧 Importación de librerías

In [2]:
import os
import gc
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm import tqdm

from ProtFlash.pretrain import load_prot_flash_base
from ProtFlash.utils import batchConverter

## ⚙️ Parámetros de extracción y hardware

In [ ]:
INPUT_CSV = "/kaggle/input/datasets/user/data-consolidation"
OUTPUT_EMBEDDINGS = "/kaggle/working/"

os.makedirs(OUTPUT_EMBEDDINGS, exist_ok=True)

L_MAX = 100
EMB_DIM = 768
batch_size=128

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

Device: cuda


## 📂 Carga de splits consolidados

In [4]:
def load_split(name):
    df = pd.read_csv(os.path.join(INPUT_CSV, f"{name}.csv"))
    data = [(row["id"], str(row["sequence"]).upper().strip()) for _, row in df.iterrows()]
    labels = df["label"].values.astype(np.int64)
    return data, labels, df["id"].tolist()

train_data, train_labels, train_ids = load_split("train")
val_data,   val_labels,   val_ids   = load_split("val")
test_data,  test_labels,  test_ids  = load_split("test")

print(f"📊 Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

📊 Train: 13826 | Val: 3614 | Test: 13762


## 🤖 Inicialización del modelo ProtFlash

In [5]:
model = load_prot_flash_base()
model.to(device)
model.eval()
print(f"✅ ProtFlash-base model loaded ({sum(p.numel() for p in model.parameters())/1e6:.0f}M params)")

Downloading: "https://zenodo.org/record/7655858/files/protflash_large.pt" to /root/.cache/torch/hub/checkpoints/protflash_large.pt
✅ ProtFlash-base model loaded (175M params)


## 🔬 Pipeline de extracción, padding y persistencia

In [6]:
def extract_per_residue_embeddings(data_split, labels, ids_split, split_name, batch_size=64):
    """
    Extract embeddings using ProtFlash's residual method, pad to L_MAX=100,
    generate a binary mask, and save as .npy
    """
    total = len(data_split)
    all_embeddings = []  # (L_max, 768)
    all_masks = []       # (L_max,)
    all_global = []      # (768,) para atrapar el Token CLS/Mean de contexto
    all_ids = []

    print(f"\n🔬 Extracting embeddings for {split_name.upper()} ({total} sequences)...")

    for i in tqdm(range(0, total, batch_size), desc=f"{split_name}"):
        batch_data = data_split[i:i + batch_size]
        ids, batch_token, lengths = batchConverter(batch_data)

        batch_token = batch_token.to(device)
        lengths = lengths.to(device)

        with torch.no_grad():
            token_embedding = model(batch_token, lengths)  # (batch, L, 768)

        token_embedding = token_embedding.cpu().numpy()  # (batch, L, 768)
        lengths = lengths.cpu().numpy()

        for j, (seq_id, seq) in enumerate(batch_data):
            seq_len = len(seq)
            
            # Manejo del Token [CLS] o de inicio
            # ProtFlash generalmente agrega un token al inicio
            # Validamos temporalmente si el length real devuelto es len(seq) + 1 o len(seq) + 2
            returned_len = lengths[j]
            offset_start = 1 if returned_len > seq_len else 0
            
            # 1. Extraemos los residuos estrictos del peptido
            emb = token_embedding[j, offset_start : offset_start + seq_len, :]  # (seq_len, 768)
            
            # 2. Extraemos información global (Si hay un [CLS] en 0 lo tomamos, sino, tomamos el promedio)
            if offset_start == 1:
                global_emb = token_embedding[j, 0, :]
            else:
                global_emb = emb.mean(axis=0)

            # Padding to L_MAX
            padded_emb = np.zeros((L_MAX, 768), dtype=np.float32)
            padded_emb[:seq_len, :] = emb

            # Binary mask: 1 for actual positions, 0 for padding
            mask = np.zeros(L_MAX, dtype=np.float32)
            mask[:seq_len] = 1.0

            all_embeddings.append(padded_emb)
            all_masks.append(mask)
            all_global.append(global_emb)
            all_ids.append(seq_id)

        del batch_token, token_embedding, ids, lengths, batch_data
        torch.cuda.empty_cache()
        gc.collect()

    # ── Saved as .npy ─────────
    X = np.stack(all_embeddings, axis=0)      # (N, L_max, 768)
    M = np.stack(all_masks, axis=0)           # (N, L_max)
    G = np.stack(all_global, axis=0)          # (N, 768)
    y = np.array(labels, dtype=np.int64)      # (N,)

    np.save(os.path.join(OUTPUT_EMBEDDINGS, f"{split_name}_embeddings.npy"), X)
    np.save(os.path.join(OUTPUT_EMBEDDINGS, f"{split_name}_masks.npy"), M)
    np.save(os.path.join(OUTPUT_EMBEDDINGS, f"{split_name}_global.npy"), G) # <-- Guardar globales
    np.save(os.path.join(OUTPUT_EMBEDDINGS, f"{split_name}_labels.npy"), y)

    pd.DataFrame({"seq_id": all_ids}).to_csv(
        os.path.join(OUTPUT_EMBEDDINGS, f"{split_name}_ids.csv"), index=False
    )

    print(f"✅ {split_name.upper()} saved:")
    print(f"   X (residues):  {X.shape}  → {split_name}_embeddings.npy")
    print(f"   G (global):    {G.shape}  → {split_name}_global.npy")
    print(f"   M (masks):     {M.shape}  → {split_name}_masks.npy")
    print(f"   y (labels):    {y.shape}  → {split_name}_labels.npy")

    return X, M, y

## 🚀 Ejecución secuencial por particiones

TRAIN

In [7]:
extract_per_residue_embeddings(train_data, train_labels, train_ids, "train", batch_size=batch_size)


🔬 Extracting embeddings for TRAIN (13826 sequences)...


train: 100%|██████████| 109/109 [03:35<00:00,  1.97s/it]


✅ TRAIN saved:
   X (residues):  (13826, 100, 768)  → train_embeddings.npy
   G (global):    (13826, 768)  → train_global.npy
   M (masks):     (13826, 100)  → train_masks.npy
   y (labels):    (13826,)  → train_labels.npy


(array([[[  3.907624  ,  -3.0747433 ,  -1.8185503 , ...,   2.0871067 ,
           -2.139746  ,   3.288681  ],
         [  1.9679772 ,   1.7797936 ,  -2.41565   , ...,  -2.4535131 ,
            3.3019319 ,   3.2214165 ],
         [  1.7192526 ,   1.85421   ,  -0.8350049 , ...,  -1.3859841 ,
           -2.9667997 ,   5.9705253 ],
         ...,
         [  0.        ,   0.        ,   0.        , ...,   0.        ,
            0.        ,   0.        ],
         [  0.        ,   0.        ,   0.        , ...,   0.        ,
            0.        ,   0.        ],
         [  0.        ,   0.        ,   0.        , ...,   0.        ,
            0.        ,   0.        ]],
 
        [[  6.9593577 ,   0.42107052,   4.922604  , ...,   1.2574911 ,
            4.7044683 ,   0.83083606],
         [  5.4698086 ,  -2.0084705 ,  -3.3665686 , ...,  -6.4371624 ,
            0.8892071 ,   0.42285255],
         [  1.2742114 ,  -3.5738387 ,   2.5618548 , ...,   2.2073696 ,
           -2.9502268 ,  -1.9324

VAL

In [8]:
extract_per_residue_embeddings(val_data,   val_labels,   val_ids,   "val",   batch_size=batch_size)


🔬 Extracting embeddings for VAL (3614 sequences)...


val: 100%|██████████| 29/29 [00:57<00:00,  2.00s/it]


✅ VAL saved:
   X (residues):  (3614, 100, 768)  → val_embeddings.npy
   G (global):    (3614, 768)  → val_global.npy
   M (masks):     (3614, 100)  → val_masks.npy
   y (labels):    (3614,)  → val_labels.npy


(array([[[ 4.389364  , -0.6830908 , -2.7782187 , ...,  0.48982862,
          -2.5420942 , -4.651147  ],
         [ 0.5597287 , -2.1982775 , -2.6502047 , ..., -0.74597186,
           0.8311892 ,  0.39836788],
         [ 3.319204  , -2.4858098 , -4.536279  , ..., -1.0997272 ,
           1.763758  , -1.5787295 ],
         ...,
         [ 0.        ,  0.        ,  0.        , ...,  0.        ,
           0.        ,  0.        ],
         [ 0.        ,  0.        ,  0.        , ...,  0.        ,
           0.        ,  0.        ],
         [ 0.        ,  0.        ,  0.        , ...,  0.        ,
           0.        ,  0.        ]],
 
        [[ 0.48425516, -4.879944  , -5.4779325 , ..., 12.995243  ,
          -0.9954408 , -2.645489  ],
         [-1.9831878 , -2.8383117 , -9.286285  , ..., 12.304274  ,
          -3.8121457 , -1.2467704 ],
         [-1.488348  , -7.0440793 , -4.430983  , ...,  6.6405354 ,
          -0.585162  , -2.2164307 ],
         ...,
         [ 0.        ,  0.       

TEST

In [9]:
extract_per_residue_embeddings(test_data,  test_labels,  test_ids,  "test",  batch_size=batch_size)


🔬 Extracting embeddings for TEST (13762 sequences)...


test: 100%|██████████| 108/108 [03:34<00:00,  1.98s/it]


✅ TEST saved:
   X (residues):  (13762, 100, 768)  → test_embeddings.npy
   G (global):    (13762, 768)  → test_global.npy
   M (masks):     (13762, 100)  → test_masks.npy
   y (labels):    (13762,)  → test_labels.npy


(array([[[  1.071662  ,  -3.0814085 ,   2.6660988 , ...,   5.276167  ,
           -0.99940336,  -4.9881024 ],
         [ -6.4726477 ,   3.668923  ,  -3.5504403 , ...,   0.25008923,
            1.9910295 ,   3.0891588 ],
         [ -0.5154985 ,   2.4617825 ,  -2.3113155 , ...,   1.2993816 ,
            0.7080061 ,  -2.1069388 ],
         ...,
         [  0.        ,   0.        ,   0.        , ...,   0.        ,
            0.        ,   0.        ],
         [  0.        ,   0.        ,   0.        , ...,   0.        ,
            0.        ,   0.        ],
         [  0.        ,   0.        ,   0.        , ...,   0.        ,
            0.        ,   0.        ]],
 
        [[  3.9381917 ,   0.18323843,  -1.7808335 , ...,   4.4567857 ,
            0.31988624,  -0.38183528],
         [ -1.8507001 ,   1.2540852 ,  -6.964824  , ...,   1.1190323 ,
            1.3549993 ,  -0.5925449 ],
         [ -0.3183067 ,  -2.0731766 ,  -7.273803  , ...,  -1.4985591 ,
           -2.4471562 ,  -1.5251

# 🏁 Resumen del Pipeline de Embeddings | ProtFlash

Este notebook cierra la fase de **vectorización semántica**, transformando secuencias de texto en tensores numéricos de alta dimensionalidad, optimizados y persistentes para consumo directo en el entrenamiento del modelo.

| 📦 Entregable | 📐 Formato | 📊 Volumen |
|:---|:---|:---|
| `*_embeddings.npy` | `(N, 100, 768)` | ✅ 31,202 tensores (train+val+test) |
| `*_masks.npy`      | `(N, 100)`      | ✅ Máscaras binarias de atención |
| `*_labels.npy`     | `(N,)`          | ✅ Etiquetas `0/1` alineadas |
| `*_ids.csv`        | `seq_id`        | ✅ Trazabilidad completa |

### 📌 Logros Clave
- **Inferencia optimizada**: Procesamiento por lotes con liberación explícita de VRAM (`torch.cuda.empty_cache()` + `gc`).
- **Estandarización dimensional**: Todas las secuencias normalizadas a `100 × 768` con padding controlado.
- **Separación estricta**: Splits mantenidos y guardados de forma independiente para evitar fugas de datos.